In [962]:
%reset -f 
# resetting stored variables in case there's something weird cached
from build123d import *
from ocp_vscode import *
import cadquery as cq
import time
import math
from library.tools import *
import sys
from dataclasses import dataclass, field
import bd_warehouse.thread, bd_warehouse.fastener 
from pathlib import Path

ALL UNITS IN MM
DON'T @ ME

In [963]:
%reload_ext ocp_vscode
%load_ext autoreload
%autoreload 2
print(f"Python executable: {sys.executable}")
print(f"OCP-vscode location: {sys.modules.get('ocp_vscode', 'Not found')}")
reset_show()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Python executable: c:\Users\Kaoti\parthenon\.venv\Scripts\python.exe
OCP-vscode location: <module 'ocp_vscode' from 'c:\\Users\\Kaoti\\parthenon\\.venv\\Lib\\site-packages\\ocp_vscode\\__init__.py'>


In [964]:
# INPUTS GO HERE

# Chamber identity
chamberIdentity = "GoliathPosterior" # Other possibilities: GoliathAnterior, MalachiRight, MalachiLeft. More TBA

# Coordinate sites / Penetration center points - these will be inputs which generate whole structure
penetration1 = (0,7.7)
penetration2 = (0,2.475)
penetration3 = (0,-1.775)
penetration4 = (0,-7)
points2D = (penetration1, penetration2, penetration3, penetration4)

# Diagnostic mode yes or no? If y, then we should set it up to have Show() commands for diagnostic purposes that turn on with a time.sleep() so we can do analysis and show off what's happening. 
diagnosticMode = 2 # user input field eventually. 0 is no, 1 is full diagnostics, 2 is timefield reporting for subfield functions only. 
smallDiagnosticTime = 0.1
largeDiagnosticTime = 0.5

Start of future basisCylinder function

In [965]:
basisCylinder = chamberCylinder(chamberIdentity,diagnosticMode)

2.646446199854836


End future basisCylinder function

In [966]:
# Importing reference chamber mesh - Goliath Posterior V3 starting V3 
# this will be an if statement based on a chamber input field. That field will also need
# to adjust the basis cylinder we generate 

if chamberIdentity == "GoliathPosterior":
    meshName = "chamberMoldMeshGoliathPosteriorV3.stl"
else:
    print("ERROR: INCOMPATIBLE CHAMBER IDENTITY. PLEASE REFACTOR INPUT.")

# meshName is going to be input from either a user field or from a chamber object which is input from a user field. either way this is close to the origin of the object, so it is upstream of lots of things. 
chamberMoldCachePath = Path(meshName).with_suffix(".brep")

if chamberMoldCachePath.exists():
    chamberMold = import_brep(chamberMoldCachePath)
else:
    chamberMold = Mesher().read(meshName)[0]
    export_brep(chamberMold,chamberMoldCachePath)

# NOTE: MESHES CANNOT BE TRANSLATED. TRANSLATE OTHER OBJECTS AROUND THEM.

# Displaying the mesh and the cylinder both
if diagnosticMode == 1:
    show_object(chamberMold, name = "chamberMold")
    print(chamberMoldCachePath)
    time.sleep(largeDiagnosticTime)

# possibility: Using a bounding circle to remove the vertical triangles on the sides of the mesh, in order to reduce triangle count and increase speed - backburnered. use trimesh?



Immediately below this point: Probably need to integrate coordinates from SPOTS somehow. Likely, also need a least-squares fit line for the arbor, and some parameter permutation if we want a normal offset from the arbor line. 

In [967]:
# Retrieving / generating points for probe penetration

# Parameters
from sympy import false
meshZ = -5
GTHoleDiameter = 0.675
recessDiameter = 3.1
GTExteriorDiameter = 2
points3D = []

for i, x in enumerate(points2D):
    points3D.append(x + (meshZ,))

circularLocations = []

# Alternative construction which leaves each point accessible
for x in points3D:
    with BuildSketch() as sk:
            with Locations(x):
                Circle(radius = GTExteriorDiameter, align=None)
    circularLocations.append(sk)

# Everybody gets a diameter
with BuildSketch() as sk:
    with Locations(points3D):
        Circle(radius = 2)
        Circle(radius = GTHoleDiameter/2, mode = Mode.SUBTRACT)

# These sketches now exist
penetrationLocations = sk

"""
    show (chamberMold, basisCylinder, circularLocations, reset_camera = Camera.RESET)  
"""

# Displaying everything at once
if diagnosticMode == 1: 
    show_object(circularLocations, name = "transient")
    print(circularLocations)
    time.sleep(largeDiagnosticTime)

In [968]:
# Mesh projected curves
# This projects circles down onto the mesh surface and adds interfaces onto list
projectedCurves = []
for x in circularLocations:
    wire = x.sketch.faces()[0].outer_wire()
    hits = wire.project_to_shape(chamberMold,direction=(0,0,-1))
    projectedCurves.append(hits)

if diagnosticMode == 1: 
    print(projectedCurves)
    show_object (projectedCurves, name = "transient", update = True)
    time.sleep(largeDiagnosticTime)

In [969]:
# Building surfaces from projected curves
# To clarify, this grabs the center point of each curve, and the slope at that point
# Then uses those slopes to construct planes. These planes are later used to contruct shafts

frameFaces = []

for x in circularLocations:
    face = x.sketch.faces()[0]
    center = face.center()
    axis = Axis((center.X, center.Y, chamberMold.bounding_box().max.Z), (0,0,-1))
    point, normal = chamberMold.find_intersection_points(axis)[-1]
    tiltedPlane = Plane(origin=point, z_dir = normal)
    frameFaces.append(face.located(Location(tiltedPlane)))

if diagnosticMode == 1: 
    print(frameFaces)
    show_object (chamberMold, name = "chamberMold", mode = Render.NONE, update = True)
    show_object (frameFaces, name = "transient", update = True)
    time.sleep(largeDiagnosticTime)

In [970]:
# Offsetting the projected curve faces to provide targets for extrusion limits
# This allows 2mm of space between the tissue surface and the the bottom of the shafts

offsetFrameFaces = []
zPlanarOffset = 2   

for x in frameFaces:
    plane1 = Plane(x)
    offsetPlane = Plane(origin = plane1.origin + (0,0,zPlanarOffset), x_dir=plane1.x_dir, z_dir = plane1.z_dir)
    offsetFrameFaces.append(offsetPlane)

if diagnosticMode == 1: 
    print(offsetFrameFaces)
    show_object (offsetFrameFaces, name = "transient", update = True)
    time.sleep(largeDiagnosticTime)


In [971]:
# Transforms the faces which the shafts start extruding on into planes so they can be useful
# Should probably look at this - not sure if this is redundant

startingOffsetPlanes = []
shaftHeight = 3.85 # This is a rough estimation based on Anna's onshape
shaftDiameter = 4
shaftRadius = shaftDiameter/2
innerShaftDiameter = 0.67500
innerShaftRadius = innerShaftDiameter/2

for x in offsetFrameFaces: # These are actually planes, not faces
    appendMe = Plane(origin = x.origin + (0,0,shaftHeight), x_dir = (1,0,0), z_dir = (0,0,1))
    startingOffsetPlanes.append(appendMe)

# Generating solids from bottom surface planes for later use.

bottomSurfaceSolids = []

for x in offsetFrameFaces:
    with BuildPart() as cyl:
        with BuildSketch(x):
            Circle(radius = shaftDiameter)
        extrude(amount=1)
    bottomSurfaceSolids.append(cyl.part)

if diagnosticMode == 1: 
    print(startingOffsetPlanes,bottomSurfaceSolids)
    show_object (startingOffsetPlanes, name = "transient", update = True)
    time.sleep(largeDiagnosticTime)
    show_object (bottomSurfaceSolids, name = "transient", update = True)
    time.sleep(largeDiagnosticTime)

In [972]:
# Building circles on the projected curves
shaftList = []
throughHoles = []
shaftListWithThroughHoles = []
seams = []
j = 0

for i, x in enumerate(startingOffsetPlanes): 
    # First build the outer shaft
    with BuildPart() as shaft:
        with BuildSketch(x) as sk:
            Circle(radius=shaftRadius)
        extrude(until=Until.NEXT, target = bottomSurfaceSolids[i], dir=(0,0,-1))
    shaftList.append(shaft)

    # Next build the throughhole
    with BuildPart() as throughhole:
        with BuildSketch(x) as sk:
            Circle(radius = innerShaftRadius)
        extrude(until = Until.NEXT, target = bottomSurfaceSolids[i], dir=(0,0,-1))
    throughHoles.append(throughhole)

    # Now subtract the throughhole geometry from the shaft geometery
    with BuildPart() as shaftWithThroughHole:
        add(shaftList[i])
        add(throughHoles[i], mode = Mode.SUBTRACT)
    shaftListWithThroughHoles.append(shaftWithThroughHole)

# show the throughholes in red and the shafts in yellow for diagnostic purposes
if diagnosticMode == 1: 
    print(startingOffsetPlanes,bottomSurfaceSolids)
    show_object (shaftList, name = "shaftList", update = True)
    time.sleep(largeDiagnosticTime)
    show_object (throughHoles, name = "transient", update = True, options={"color": (255, 0,0)})
    time.sleep(largeDiagnosticTime)

In [973]:
# Fillet slanted bottom edge of shaft and throughhole
targetFaces = []
filletItems = []
targetEdgesMasterList = []

for i, x in enumerate(shaftListWithThroughHoles):
    targetEdges = []
    targetFace = min(shaftListWithThroughHoles[i].faces(), key = findZ)
    targetFaces.append(targetFace)
    targetEdge1, targetEdge2 = targetFace.edges()
    targetEdges.append(targetEdge1)
    targetEdges.append(targetEdge2)
    filletItem: Sketch | Part | Curve = fillet(targetEdges, radius = 0.20)
    filletItems.append(filletItem)
    targetEdgesMasterList.append(targetEdges)

shaftListWithThroughHolesFilleted = filletItems

if diagnosticMode == 1: 
    print(targetEdges,shaftListWithThroughHolesFilleted)
    show_object (targetEdgesMasterList, name = "transient", update = True)
    time.sleep(largeDiagnosticTime)
    show_object (shaftList, name = "shaftList", mode = Render.NONE, update = True)
    show_object (shaftListWithThroughHolesFilleted, name = "transient", update = True)
    time.sleep(largeDiagnosticTime)

In [974]:
# Fillet-ing top edge of throughholes
filletList = []

for i, x in enumerate(shaftListWithThroughHolesFilleted):
    edge = x.edges().filter_by(GeomType.CIRCLE).filter_by(
        lambda a: abs(a.radius - innerShaftRadius) < 1e-6)
    seams.append(edge)

    filletObject = fillet(edge, radius = 0.25)
    filletList.append(filletObject)

show(filletList, reset_camera = Camera.RESET)
shaftListWithThroughHolesFilletedTwice = filletList


cccc


In [975]:
# Display everything for the purpose of sanity checks

show(basisCylinder, shaftListWithThroughHolesFilletedTwice, chamberMold, alphas=[1.0,1.0,0.8], colors=["#e8b024", "#e8b024", "lightblue"], reset_camera=Camera.RESET)

cccccc


Okay, the actual through holes for the GT have been implanted in the shafts. Location-dependant shaft placement and homing has been completed in a scalable fashion. Now we need to connect the shafts to the basis cylinder. 

In [976]:
# GT Depth Calculator: To-do

In [977]:
# Nubs
# Doing this a little bit differently from Anna's Onshape. Instead of building in the default plane, I'm going to build each set of nubs on the top plane of the actual shaft it's attached to.

parallelNubSeparation = 2.64575
nubLongSide = 3
nubShortSide = 1.35425
nubDepth = 2.5
nubsList = []
rotatorLocations = [1,2,3,4]
rotatorAngle = 90
overlaps = []
nubTemplates = []
nubTemplatesFlattened = []

nubDisplacementVector = (0,parallelNubSeparation/2 + nubShortSide/2,0)
nubRotationVector = (0,0,1)

for i, x in enumerate(startingOffsetPlanes):
    for k, j in enumerate(rotatorLocations):
        nubTemplate = Rectangle(nubLongSide,nubShortSide).located(Location(startingOffsetPlanes[i])).translate(nubDisplacementVector).rotate(
            axis = Axis(startingOffsetPlanes[i].origin, nubRotationVector),
            angle = rotatorAngle*k
        )
        nubTemplates.append(nubTemplate)
        with BuildPart() as nubs:
            extrude(nubTemplate, amount=nubDepth, dir = (0,0,-1))
        nubsList.append(nubs.part)

for i, x in enumerate(nubTemplates):
    nubTemplatesFlattened.append(flattenToXY(nubTemplates[i]))

# Overlap areas between nubs 2,4
overlaps.append(nubTemplatesFlattened[2] & nubTemplatesFlattened[4])

# Overlap areas between nubs 6,8
overlaps.append(nubTemplatesFlattened[6] & nubTemplatesFlattened[8])

# Overlap areas between nubs 8,10
overlaps.append(nubTemplatesFlattened[10] & nubTemplatesFlattened[12])

print(overlaps)

show(basisCylinder, shaftListWithThroughHolesFilletedTwice, nubsList, overlaps, reset_camera=Camera.RESET)


[Compound at 0x2111b1a7c80, label(), #children(0), Compound at 0x2111b1a62a0, label(), #children(0), Compound at 0x21108f78050, label(), #children(0)]
cccccccccccccccccccccccc


In [978]:
# Constructing overlap solids, pruning union parts between different shafts

overlapSolids = []
prunedParts = []
lofts = []

# Generating overlap structures
for i, x in enumerate(overlaps):
    with BuildPart() as firstpart:
        extrude(overlaps[i], amount = shaftHeight*10, dir = (0,0,-1))
        overlapSolids.append(firstpart.part)

# Deleting overlap between shafts 1 and 2, numbered from +y to -y in global coordinates
with BuildPart() as pt:
    add (nubsList[2])
    add (overlapSolids[0], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

with BuildPart() as pt:
    add (nubsList[4])
    add (overlapSolids[0], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

loft1 = loftMe(nubsList[2], nubsList[4])
lofts.append(loft1)

# Deleting overlap between shafts 10 and 12, numbered from +y to -y in global coordinates
with BuildPart() as pt:
    add (nubsList[10])
    add (overlapSolids[2], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

with BuildPart() as pt:
    add (nubsList[12])
    add (overlapSolids[2], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

loft2 = loftMe(nubsList[10], nubsList[12])
lofts.append(loft2)

with BuildPart() as pt:
    if startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
        add (nubsList[6])
    elif startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
        add (nubsList[8])
    else:
        add (nubsList[6])
        add (nubsList[8])
    add(overlapSolids[1], mode=Mode.SUBTRACT)
    prunedParts.append(pt.part)

show(prunedParts, reset_camera=Camera.RESET, colors=["#e8b024", "#e8b024", "#e8b024",  "lightblue"])


Too many colors, trimming to length 1
ccccc


In [979]:
show(prunedParts[-1])
len(prunedParts)

c


5

In [980]:
# Combining nubs and shafts with throughholes

nubulousShaftsWithThroughHoles = []
iteratorNumbers = []

for i, x in enumerate(shaftListWithThroughHolesFilletedTwice):
    with BuildPart() as NubulousShaftWithThroughHole:
        for k, j in enumerate(nubsList):
            iteratorNumber = k // 4
            if iteratorNumber == i and j != 6 and j != 8:
                add(nubsList[k])
            elif iteratorNumber == i and j == 6 or iteratorNumber == i and j == 8:
                if startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
                    add (nubsList[6])
                elif startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
                    add (nubsList[8])
                else:
                    add (nubsList[6])
                    add (nubsList[8])
            else:
                continue
        add(shaftListWithThroughHolesFilletedTwice[i])
        add(overlapSolids, mode = Mode.SUBTRACT)
    nubulousShaftsWithThroughHoles.append(NubulousShaftWithThroughHole.part)

show(nubulousShaftsWithThroughHoles, reset_camera=Camera.RESET)




# Pseudocode
# add the 4 adjacent nubs to their shaft
    # add the shaft i
    # add the 4 nubs whose floor division is equal to i
    # Combine these into single part
# fillet the intersections
# append these combined units to a new list

cccc


In [981]:
# type Part = build123d.topology.composite.Part
from build123d.topology.composite import Part

In [982]:
def printPart(part: Part):
    print(part)

printPart(nubsList[0])
printPart(3)

Part at 0x210c3f2fc80, label(), #children(0)
3


In [983]:
print(type(nubsList[1]))

<class 'build123d.topology.composite.Part'>


In [984]:
# Generating positional arms to connect nubs and basis cylinder

testArms = []
outputArmStream = []
refinedNubsList = [nubsList[1], nubsList[5], nubsList[9], nubsList[13]]

for i, x in enumerate(refinedNubsList):
    arm1, mirrorArm,armTest = armConstructor(i, refinedNubsList[i],startingOffsetPlanes[i])
    outputArmStream.append(arm1)
    outputArmStream.append(mirrorArm)
    testArms.append(armTest)
    # Ensure construction sketches are either nub-centric, or somehow related directionally to the rotation of the arbor centerline. 
show(basisCylinder, shaftListWithThroughHolesFilletedTwice, nubsList, outputArmStream, colors=["#e8b024", "#e8b024", "#e8b024",  "lightblue", "pink", "pink"])



Too many colors, trimming to length 4
ccccccccccccccccccccccccccccc


In [985]:
# probably make this a function
for i, x in enumerate(faceList):
    show(faceList[i], targetZFace[zIndicies[3]])

NameError: name 'faceList' is not defined

In [ ]:
# Site-based data storage 

@dataclass
class Site:
    shaft: Part
    plane: Plane
    nubs: list
    arms: list = field(default_factory = list)
    outer: object = None
    throughHole: object = None
    combined: object = None

# Establishing the variables used in this cell
sites = []
planeHeightIndicator = 0

# handles the x-axial nubs and the outer union parts, plus the shafts
for i, plane in enumerate(startingOffsetPlanes):
    sites.append(Site(
        shaft = shaftListWithThroughHolesFilletedTwice[i],
        plane = plane,
        nubs = nubsList[i*4+1:i*4 + 4:2] + [prunedParts[i]],
        arms = outputArmStream[i*2:i*2 + 2]
    )) 

# Picks the highest(z) starting offset plane of the two middle planes. 
# Adds the central nub of the higher z site and appends to nubs
# trims the central nub of the lower z site and appends that to nubs
if startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
    sites[1].nubs.append((nubsList[6]))
    sites[2].nubs.append((prunedParts[-1]))
    planeHeightIndicator = 2 # this keeps track of the higher plane
elif startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
    sites[2].nubs.append((nubsList[8]))
    sites[1].nubs.append((prunedParts[-1]))
    planeHeightIndicator = 1 # this keeps track of the higher plane

# covers the case in which the zaxis is the same, in which case we just add both nubs
else:
    sites[1].nubs.append((nubsList[6]))
    sites[2].nubs.append((nubsList[8]))

show(*[site.shaft for site in sites], [site.nubs for site in sites], [site.arms for site in sites])

cccccccccccccccccccccccccc


In [ ]:
"""

# Constructing final solid part
with BuildPart() as fusedPart:

    # Shafts and side nubs
    for i, x in enumerate(shaftListWithThroughHolesFilletedTwice):
        add (shaftListWithThroughHolesFilletedTwice[i]), 
        add (nubsList[1::2])
        
    # separable npx pairs need a union. this is the math for that union.
    add (prunedParts[4])
    if startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
        add (nubsList[6])
    elif startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
        add (nubsList[8])
    else:
        add (nubsList[6], nubsList[8])
    
    # paired npx nubs
    add (prunedParts[0])
    add (prunedParts[1])
    add (loft1)

    add (prunedParts[2])
    add (prunedParts[3])
    add (loft2)

with BuildPart() as anotherpart:
    # Basis cylinder
    add (basisCylinder)
    add (fusedPart)

    # adding arms
    add (outputArmStream)
    


guideTubeFrameNoChamfer = anotherpart
show(guideTubeFrameNoChamfer, basisCylinder, reset_camera=Camera.RESET)
"""

'\n\n# Constructing final solid part\nwith BuildPart() as fusedPart:\n\n    # Shafts and side nubs\n    for i, x in enumerate(shaftListWithThroughHolesFilletedTwice):\n        add (shaftListWithThroughHolesFilletedTwice[i]), \n        add (nubsList[1::2])\n\n    # separable npx pairs need a union. this is the math for that union.\n    add (prunedParts[4])\n    if startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:\n        add (nubsList[6])\n    elif startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:\n        add (nubsList[8])\n    else:\n        add (nubsList[6], nubsList[8])\n\n    # paired npx nubs\n    add (prunedParts[0])\n    add (prunedParts[1])\n    add (loft1)\n\n    add (prunedParts[2])\n    add (prunedParts[3])\n    add (loft2)\n\nwith BuildPart() as anotherpart:\n    # Basis cylinder\n    add (basisCylinder)\n    add (fusedPart)\n\n    # adding arms\n    add (outputArmStream)\n\n\n\nguideTubeFrameNoChamfer = anotherpart\nshow(guideTubeFram

In [ ]:
# Sitewise construction using dataclasses

fillets = []
filletedItems = []


for i, site in enumerate(sites):
    with BuildPart() as combinedSite:
        add(site.shaft)
        add(site.nubs)
    filletMe = new_edges(
        site.shaft, 
        *site.nubs, 
        combined = combinedSite.part).filter_by(GeomType.LINE)
    fillets.extend(filletMe)
    if combinedSite.part.max_fillet(filletMe) > 0.25:
        filletParameter = 0.25
    else:
        filletParameter = combinedSite.part.max_fillet(filletMe)

    filletedItem = fillet(filletMe, radius = filletParameter)
    
    filletedItems.append(filletedItem)



show(filletedItems)

cccc


In [ ]:
# Adding arms sitewise
combinedSitesWithArms = []


for i, site in enumerate(sites):
    with BuildPart() as combinedSiteWithArms:
        add(filletedItems[i])
        add(site.arms)
        combinedSitesWithArms.append(combinedSiteWithArms.part)

show([combinedSitesWithArms for site in sites])


------------cc+c


In [ ]:
# Merging with BasisCylinder

objectsInProgress = [basisCylinder]
armFillets = []

for i, site in enumerate(sites):
    with BuildPart() as newPart:
        add(objectsInProgress[i])
        add(combinedSitesWithArms[i])
        armFillet = new_edges(
            combinedSitesWithArms[i],
            objectsInProgress[i],
            combined = newPart.part).filter_by(GeomType.LINE).filter_by(lambda e: e.length >= 2.25)
        objectsInProgress.append(fillet(armFillet, radius = 0.15))
        armFillets.append(armFillet)

show(*[armFillets], objectsInProgress[-1])
armFillet[2].length


+


2.5

In [ ]:
# adding lofts to make final part  
objectsInProgress2 = list(objectsInProgress)
loftFillets = []

with BuildPart() as nextPart:
    add (objectsInProgress2[-1])
    add (loft1)
    add (loft2)
    """loftFillet = new_edges(
        loft,
        objectsInProgress2[-1],
        combined = newPart.part).filter_by(GeomType.LINE)"""
    show(objectsInProgress2,loft)
    loftFillet = nextPart.edges().filter_by(
        GeomType.LINE).filter_by(
        lambda e: abs(e.length - 3) < 1e-6).filter_by(
        lambda e: abs(e.position_at(0).Z - e.position_at(1).Z) < 1e-6).filter_by(
        lambda e: abs(e.position_at(0).Y - e.position_at(1).Y) < 1e-6).filter_by(
        lambda e: e.center().Z < -1)
    loftFillets.extend(loftFillet)
    objectsInProgress.append(nextPart.part)
    print(loftFillets)
    objectsInProgress2.append(fillet(loftFillet, radius = 0.1))
        
finalPart = objectsInProgress2[-1]
show(loftFillets, finalPart)

c++++
[<build123d.topology.one_d.Edge object at 0x00000210C3BF9400>, <build123d.topology.one_d.Edge object at 0x00000210C3BFB700>, <build123d.topology.one_d.Edge object at 0x00000210C3BFAEB0>, <build123d.topology.one_d.Edge object at 0x00000210C3B94520>, <build123d.topology.one_d.Edge object at 0x0000021108E1DC50>, <build123d.topology.one_d.Edge object at 0x000002111B202BA0>, <build123d.topology.one_d.Edge object at 0x0000021108ED6660>, <build123d.topology.one_d.Edge object at 0x000002111B135F60>]
c


In [ ]:
show(loftFillets, objectsInProgress2[-1])

c


In [ ]:
for i, e in enumerate(loftFillet):
    try:
        r = nextPart.part.max_fillet([e])
        print(i, "ok — max radius:", r)
    except Exception as err:
        print(i, "BAD EDGE:", type(err).__name__, err)


0 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
1 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
2 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
3 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
4 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
5 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
6 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
7 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet


In [ ]:

print(len(nextPart.part.solids()))


1


In [ ]:
export_stl(finalPart, "ParametricGuideTubeFrame.stl")

True